In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "ethiopia"
vehicle = "salt"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention_25_nrv',
 'intervention_100_nrv',
 'intervention_45_ppm',
 'zero',
 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention_25_nrv,0,141,0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention_25_nrv,0,141,0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,intervention_25_nrv,0,141,0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,intervention_25_nrv,0,141,0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,intervention_25_nrv,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
538195,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,54,0
538196,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,54,0
538197,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,54,0
538198,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,54,0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline                200
intervention_100_nrv    200
intervention_25_nrv     200
intervention_45_ppm     200
zero                    200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario              entity              wealth_quintile
baseline              maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_25_nrv   maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention_25_nrv,0,141,0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention_25_nrv,0,141,0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention_25_nrv,0,141,0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention_25_nrv,0,141,0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,intervention_25_nrv,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
1883695,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,54,0
1883696,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,54,0
1883697,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,54,0
1883698,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,54,0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario              entity              wealth_quintile
baseline              anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
                      maternal_disorders  1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                          5                  0.0
intervention_100_nrv  anemia              1                  0.0
                                          2                  0.0
                                          3                  0.0
                                          4                  0.0
                                

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # NOTE: This else branch is for processing the Ethiopia results,
    # where no Vivarium sims were run, so the corresponding DALYs should
    # just be 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention_25_nrv,0,81,0
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention_25_nrv,0,81,0
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention_25_nrv,0,81,0
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention_25_nrv,0,81,0
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention_25_nrv,0,81,0
...,...,...,...,...,...,...,...,...,...,...,...,...
23955,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,0,129,0
23956,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,0,129,0
23957,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,0,129,0
23958,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,0,129,0


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario              entity  wealth_quintile
baseline              lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_100_nrv  lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_25_nrv   lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0
                              4                  0.0
                              5                  0.0
intervention_45_ppm   lbwsg   1                  0.0
                              2                  0.0
                              3                  0.0


In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,384.312386,zero
1,Female,0.0,0.019178,2,285.150179,zero
2,Female,0.0,0.019178,3,230.360898,zero
3,Female,0.0,0.019178,4,221.901969,zero
4,Female,0.0,0.019178,5,143.476230,zero
...,...,...,...,...,...,...
1245,Male,95.0,125.000000,4,24.739751,intervention_25_nrv
1246,Male,95.0,125.000000,4,24.739751,intervention_45_ppm
1247,Male,95.0,125.000000,5,17.905273,intervention_100_nrv
1248,Male,95.0,125.000000,5,17.905273,intervention_25_nrv


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  91111.681235
                              2                  60131.253431
                              3                  54905.170561
                              4                  50638.790629
                              5                  47146.871152
intervention_100_nrv  anemia  1                  71049.968031
                              2                  42830.672003
                              3                  36195.280292
                              4                  42281.496500
                              5                  40293.650844
intervention_25_nrv   anemia  1                  85390.868372
                              2                  54926.744852
                              3                  49055.046213
                              4                  48347.385890
                              5                  45304.419630
intervention_45_ppm   an

In [18]:
scenarios[1]

'intervention_100_nrv'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

71282.69933736662

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  136428.461978
                              2                   94238.537685
                              3                   70232.849032
                              4                   68652.480525
                              5                   42950.734859
intervention_100_nrv  anemia  1                  136428.461978
                              2                   94238.537685
                              3                   70232.849032
                              4                   68652.480525
                              5                   42950.734859
intervention_25_nrv   anemia  1                  136428.461978
                              2                   94238.537685
                              3                   70232.849032
                              4                   68652.480525
                              5                   42950.734859
intervent

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario              entity  wealth_quintile
baseline              anemia  1                  382307.364328
                              2                  245090.682900
                              3                  200413.088043
                              4                  193901.557471
                              5                  136614.552420
intervention_100_nrv  anemia  1                  351927.091804
                              2                  220068.193406
                              3                  173728.547134
                              4                  182040.961285
                              5                  127454.196682
intervention_25_nrv   anemia  1                  373618.791547
                              2                  237559.323759
                              3                  192079.584660
                              4                  190650.467357
                              5                  134154.158936
intervent

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario              entity  wealth_quintile
zero                  ntd     1                  155336.669764
                              2                  147760.703639
                              3                  134488.080816
                              4                  118513.950693
                              5                   93495.643956
baseline              ntd     1                  155336.669764
                              2                  147760.703639
                              3                  134488.080816
                              4                  118513.950693
                              5                   93495.643956
intervention_25_nrv   ntd     1                   73466.669878
                              2                   65558.006231
                              3                   53110.068696
                              4                   79351.290354
                              5                   70637.721418
intervent

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario  entity  wealth_quintile
baseline  anemia  1                  382307.364328
                  2                  245090.682900
                  3                  200413.088043
                  4                  193901.557471
                  5                  136614.552420
                                         ...      
zero      ntd     1                  155336.669764
                  2                  147760.703639
                  3                  134488.080816
                  4                  118513.950693
                  5                   93495.643956
Name: value, Length: 100, dtype: float64

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)